In [0]:
%pip install --quiet --upgrade unitycatalog-ai
%restart_python

In [0]:
%run ../Includes/Lab_Setup_02

In [0]:
%sql
SELECT * FROM smartphones

In [0]:
%sql
SELECT * FROM inventory_stock

In [0]:
%sql
CREATE OR REPLACE FUNCTION get_latest_phone()
RETURNS TABLE(phone_id INT, model STRING, color STRING, price INT)
COMMENT 'Returns the latest phone, ordered by release_date descending. IMPORTANT: This function accepts ZERO parameters. Do not pass any arguments.'
RETURN
  SELECT phone_id, model, color, price
  FROM smartphones
  ORDER BY release_date DESC
  LIMIT 1;

In [0]:
%sql
SELECT * FROM get_latest_phone()

In [0]:
%sql
CREATE OR REPLACE FUNCTION check_inventory(phone_id INT COMMENT 'ID of the phone to check (integer)')
RETURNS TABLE (
  item_id INT,
  quantity INT
)
COMMENT 'Checks the stock quantity for a given phone in the inventory.'
RETURN
  SELECT item_id, quantity
  FROM inventory_stock
  WHERE item_id = phone_id;

In [0]:
%sql
SELECT * FROM check_inventory(4)

In [0]:
%sql
CREATE OR REPLACE FUNCTION get_current_user()
RETURNS STRING
COMMENT 'Returns the current session user, used for identifying the active user. IMPORTANT: This function accepts ZERO parameters. Do not pass any arguments.'
RETURN current_user();

In [0]:
%sql
SELECT get_current_user()

In [0]:
def generate_checkout_url(user_id: str, phone_id: int, valid_quantity: int) -> str:
    """
    Returns the checkout URL to fulfill the user's order for a phone with a given quantity.
    The URL is valid for 30 minutes.

    Args:
        user_id (str): Identifier of the current user/session.
        phone_id (int): ID of the phone to buy.
        valid_quantity (int): the requested quantity, validated against current stock

    Returns:
        str: Checkout URL string.
    """
    from datetime import datetime, timedelta
    from urllib.parse import urlencode

    expiry_time = datetime.now() + timedelta(minutes=30)
    params = {
        "i": phone_id,
        "q": valid_quantity,
        "u": user_id,
        "t": expiry_time
    }
    
    # Return checkout URL
    checkout_url = f"https://www.derar.cloud/checkout?{urlencode(params)}"
    return checkout_url

In [0]:
generate_checkout_url("derar@derar.cloud", 4, 2)

In [0]:
from unitycatalog.ai.core.databricks import DatabricksFunctionClient

client = DatabricksFunctionClient()

python_tool_uc_info = client.create_python_function(func=generate_checkout_url,
                                                    catalog=course.catalog,
                                                    schema=course.schema,
                                                    replace=True)